# 01 — Ingest, clean, enrich (Part 1)



In [7]:
import os, csv, json, time
from pathlib import Path
import requests
import numpy as np
import pandas as pd



## Task 1.1 — CSV foundations



In [8]:
with open('/appointments.csv', newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    print('Header:', reader.fieldnames)
    raw_rows = list(reader)
print('Row count:', len(raw_rows))

def parse_row(row):
    converted = row.copy()
    for column in ['age', 'fee_inr', 'prior_no_shows']:
        converted[column] = int(row[column])
    converted['distance_km'] = float(row['distance_km'])
    return converted

accepted, rejected = [], []
for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row.copy())
print('Accepted:', len(accepted), '| Rejected:', len(rejected))
with open('rejected_rows.json', 'w', encoding='utf-8') as handle:
    json.dump(rejected, handle, indent=2, ensure_ascii=False)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted: 988 | Rejected: 32


## Tasks 1.2–1.3 — pandas cleaning


In [21]:
df = pd.read_csv('/appointments.csv')
print('Raw shape:', df.shape)
df.info()
print('Nulls:\n', df.isnull().sum())
print('Exact duplicates:', df.duplicated().sum())

# 1.2: remove exact duplicate rows
df = df.drop_duplicates().copy()
print('After duplicate removal:', len(df))
# normalize channel labels
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print('Channel labels:', sorted(df['booking_channel'].dropna().unique()))
# parse dates
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'])
# remove missing appointment date or status
df = df.dropna(subset=['appt_date', 'status']).copy()
print('After dropping missing appt_date/status:', len(df))

# 1.3a: invalid ages become missing
age_num = pd.to_numeric(df['age'], errors='coerce')
invalid_age = (age_num < 1) | (age_num > 100)
print('Impossible ages:', int(invalid_age.sum()))
df.loc[invalid_age, 'age'] = np.nan
# make numeric columns numeric; malformed values become missing
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['distance_km'] = pd.to_numeric(df['distance_km'], errors='coerce')
df['fee_inr'] = pd.to_numeric(df['fee_inr'], errors='coerce')
# fill gaps using medians
df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())
print('Age median used; distance median used')

# cap outliers using 1.5*IQR fences
for col in ['distance_km', 'fee_inr']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    count_capped = int(((df[col] < lower) | (df[col] > upper)).sum())
    df[col] = np.clip(df[col], lower, upper)
    print(col, 'Q1=', q1, 'Q3=', q3, 'lower=', lower, 'upper=', upper, 'capped=', count_capped)
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)

Raw shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
Nulls:
 appointment_id      0
patient_id          0
clinic_id         

## Task 1.4 — features and NumPy summary

In [ ]:
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)
X_summary = df[['age','distance_km','fee_inr','lead_days']].to_numpy()
print('Column means [age, distance, fee, lead]:', np.mean(X_summary, axis=0))
mask = df['lead_days'].to_numpy() > 14
print('Lead >14 count:', int(mask.sum()), 'share:', float(mask.mean()))
x = df['distance_km'].to_numpy()
z = (x - np.mean(x)) / np.std(x)
print('Distance |z| > 2:', int((np.abs(z) > 2).sum()))
print('No-shows:', int(df['no_show'].sum()), '/', len(df), 'rate:', round(100*df['no_show'].mean(),1), '%')

Column means [age, distance, fee, lead]: [ 48.17373737   9.50090909 635.71414141  11.81919192]
Lead >14 count: 296 share: 0.298989898989899
Distance |z| > 2: 53
No-shows: 218 / 990 rate: 22.0 %


## Task 1.5 — enrich clinics using Open-Meteo



In [13]:
clinics = pd.read_csv('/clinics.csv')
temps = []
for _, clinic in clinics.iterrows():
    temp = None
    reason = None
    params = {'latitude': clinic['latitude'], 'longitude': clinic['longitude'],
              'hourly': 'temperature_2m', 'forecast_days': 1}
    for attempt in range(2):
        try:
            r = requests.get('https://api.open-meteo.com/v1/forecast', params=params, timeout=10)
            if r.status_code != 200:
                reason = f'HTTP {r.status_code}'
                continue
            payload = r.json()
            values = payload.get('hourly', {}).get('temperature_2m')
            if not values:
                reason = 'hourly temperature missing'
                continue
            temp = values[0]
            break
        except requests.exceptions.RequestException as exc:
            reason = str(exc)
    if temp is None:
        print('Temperature unavailable for', clinic['clinic_name'], ':', reason)
    temps.append(temp)
clinics['temp_c'] = temps
clinics.to_csv('clinics_enriched.csv', index=False)
print(clinics)

Temperature unavailable for Kothrud Family Clinic : HTTPSConnectionPool(host='api.open-meteo.com', port=443): Read timed out. (read timeout=10)
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0     NaN  
1    19.7  
2    19.7  
3    16.4  
4    16.4  
5    26.3  
6    16.1  
7    14.7  


## Task 1.6 — final CSV and SQL load script



In [20]:

import pandas as pd


df["booked_on"] = pd.to_datetime(df["booked_on"])
df["appt_date"] = pd.to_datetime(df["appt_date"])

df["lead_days"] = (
    df["appt_date"] - df["booked_on"]
).dt.days

df["no_show"] = (
    df["status"].str.strip() == "No-Show"
).astype(int)

df = df.sort_values("appointment_id")

df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

appointments_clean = df[columns].copy()

print("Unique appointment IDs:",
      appointments_clean["appointment_id"].nunique())

print("Total appointment rows:", len(appointments_clean))

print("Missing values:")
print(appointments_clean.isnull().sum())

assert appointments_clean["appointment_id"].is_unique
assert not appointments_clean.isnull().any().any()

appointments_clean.to_csv(
    "appointments_clean.csv",
    index=False
)

clinics = pd.read_csv("clinics_enriched.csv")

def sql_value(value):

    if pd.isna(value):
        return "NULL"

    elif isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"

    else:
        return str(value)

with open("load_data.sql", "w", encoding="utf-8") as handle:
    for _, row in clinics.iterrows():
        values = [
            sql_value(row[column])
            for column in [
                "clinic_id", "clinic_name", "city",
                "tier", "latitude", "longitude", "temp_c"
            ]
        ]

        statement = (
            "INSERT INTO clinics VALUES ("
            + ",".join(values)
            + ");\n"
        )
        handle.write(statement)

    for _, row in appointments_clean.iterrows():
        values = [
            sql_value(value)
            for value in row.tolist()
        ]

        statement = (
            "INSERT INTO appointments VALUES ("
            + ",".join(values)
            + ");\n"
        )

        handle.write(statement)


print("appointments_clean.csv created!")
print("load_data.sql created!")
print("Total SQL INSERT statements:", len(clinics) + len(appointments_clean))


Unique appointment IDs: 990
Total appointment rows: 990
Missing values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
appointments_clean.csv created!
load_data.sql created!
Total SQL INSERT statements: 998
